In [1]:
# ============================================================
# 1. Importar librerías
# ============================================================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [2]:
# 2. Cargar los datos
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
url = '/content/drive/MyDrive/nacimientos/Serie_Nacimientos_2001_2019.csv'
df = pd.read_csv(url, sep=";")
df.head()

,MES_NAC,ANO_NAC,SEXO,TIPO_PARTO,TIPO_ATEN,PARTO_LOCAL,SEMANAS,RANGO_PESO,TALLA,GRUPO_ETARIO_PADRE,...,GRUPO_ETARIO_MADRE,EST_CIV_MADRE,CURSO_MADRE,NIVEL_MADRE,ACTIV_MADRE,OCUPA_MADRE,CATEG_MADRE,NACIONALIDAD_MADRE,REGION_RESIDENCIA,GLOSA_REGION_RESIDENCIA
0,1,2001,1,1,1,1,NaN,1500 - 2499,46.0,30 A 34 AÑOS,...,30 A 34 AÑOS,2.0,4.0,1.0,1.0,3,4.0,C,13.0,Metropolitana de Santiago
1,1,2001,1,1,1,1,NaN,3000 - 3999,50.0,30 A 34 AÑOS,...,30 A 34 AÑOS,2.0,4.0,2.0,0.0,2,0.0,C,13.0,Metropolitana de Santiago
2,1,2001,1,1,1,1,NaN,3000 - 3999,53.0,30 A 34 AÑOS,...,30 A 34 AÑOS,2.0,4.0,2.0,0.0,2,0.0,C,13.0,Metropolitana de Santiago
3,1,2001,1,1,1,1,24.0,<1500,29.0,25 A 29 AÑOS,...,15 A 19 AÑOS,1.0,2.0,2.0,0.0,3,0.0,C,13.0,Metropolitana de Santiago
4,1,2001,1,1,1,1,24.0,<1500,30.0,40 A 44 AÑOS,...,35 A 39 AÑOS,2.0,2.0,2.0,0.0,2,0.0,C,3.0,De Atacama


In [4]:
# 3. Exploración básica
print(df.info())
print(df.describe())
print(df.columns)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2996498 entries, 0 to 2996497
Data columns (total 25 columns):
 #   Column                   Dtype  
---  ------                   -----  
 0   MES_NAC                  int64  
 1   ANO_NAC                  int64  
 2   SEXO                     int64  
 3   TIPO_PARTO               int64  
 4   TIPO_ATEN                int64  
 5   PARTO_LOCAL              int64  
 6   SEMANAS                  float64
 7   RANGO_PESO               object 
 8   TALLA                    float64
 9   GRUPO_ETARIO_PADRE       object 
 10  CURSO_PADRE              float64
 11  NIVEL_PADRE              float64
 12  ACTIV_PADRE              float64
 13  OCUPA_PADRE              object 
 14  CATEG_PADRE              int64  
 15  GRUPO_ETARIO_MADRE       object 
 16  EST_CIV_MADRE            float64
 17  CURSO_MADRE              float64
 18  NIVEL_MADRE              float64
 19  ACTIV_MADRE              float64
 20  OCUPA_MADRE              object 
 21  CATEG_MA

In [5]:
# ============================================================
# 2. Cargar dataset
# ============================================================
# Cambia el nombre del archivo y el separador si es necesario
print("Columnas del dataset:")
print(df.columns)
print("\nPrimeras filas:")
print(df.head())

Columnas del dataset:
Index(['MES_NAC', 'ANO_NAC', 'SEXO', 'TIPO_PARTO', 'TIPO_ATEN', 'PARTO_LOCAL',
       'SEMANAS', 'RANGO_PESO', 'TALLA', 'GRUPO_ETARIO_PADRE', 'CURSO_PADRE',
       'NIVEL_PADRE', 'ACTIV_PADRE', 'OCUPA_PADRE', 'CATEG_PADRE',
       'GRUPO_ETARIO_MADRE', 'EST_CIV_MADRE', 'CURSO_MADRE', 'NIVEL_MADRE',
       'ACTIV_MADRE', 'OCUPA_MADRE', 'CATEG_MADRE', 'NACIONALIDAD_MADRE',
       'REGION_RESIDENCIA', 'GLOSA_REGION_RESIDENCIA'],
      dtype='object')

Primeras filas:
   MES_NAC  ANO_NAC  SEXO  TIPO_PARTO  TIPO_ATEN  PARTO_LOCAL  SEMANAS  \
0        1     2001     1           1          1            1      NaN   
1        1     2001     1           1          1            1      NaN   
2        1     2001     1           1          1            1      NaN   
3        1     2001     1           1          1            1     24.0   
4        1     2001     1           1          1            1     24.0   

    RANGO_PESO  TALLA GRUPO_ETARIO_PADRE  ...  GRUPO_ETARIO_MADR

In [11]:
# Antes de definir X e y
df_muestra = df.sample(n=50000, random_state=42)  # ajusta el número según tu PC
df = df_muestra

In [12]:
# ============================================================
# 3. Definir variable objetivo (y) y eliminar sus NaN
# ============================================================
objetivo = "TIPO_PARTO"

if objetivo not in df.columns:
    raise ValueError("No se encuentra la columna 'TIPO_PARTO' en el dataset.")

# Eliminar filas sin tipo de parto
df = df.dropna(subset=[objetivo])

# Convertir a entero por seguridad (si viene como float/objeto)
df[objetivo] = df[objetivo].astype(int)

print("\nDistribución de TIPO_PARTO:")
print(df[objetivo].value_counts())




Distribución de TIPO_PARTO:
TIPO_PARTO
1    48880
2     1007
9       82
3       29
4        2
Name: count, dtype: int64


In [13]:

# ============================================================
# 4. Definir variables predictoras (X)
# ============================================================
# Columnas numéricas (puedes ajustar)
columnas_numericas = [
    "SEMANAS",
    "TALLA"
]

# Columnas categóricas (basado en tu ejemplo, ajusta si falta alguna)
columnas_categoricas = [
    "MES_NAC",
    "ANO_NAC",
    "SEXO",
    "TIPO_ATEN",
    "PARTO_LOCAL",
    "RANGO_PESO",
    "GRUPO_ETARIO_PADRE",
    "CURSO_PADRE",
    "NIVEL_PADRE",
    "ACTIV_PADRE",
    "OCUPA_PADRE",
    "CATEG_PADRE",
    "GRUPO_ETARIO_MADRE",
    "EST_CIV_MADRE",
    "CURSO_MADRE",
    "NIVEL_MADRE",
    "ACTIV_MADRE",
    "OCUPA_MADRE",
    "CATEG_MADRE",
    "NACIONALIDAD_MADRE",
    "REGION_RESIDENCIA",
    "GLOSA_REGION_RESIDENCIA"
]

# Filtrar solo columnas que realmente existan en el df (por seguridad)
columnas_numericas = [c for c in columnas_numericas if c in df.columns]
columnas_categoricas = [c for c in columnas_categoricas if c in df.columns]

X = df[columnas_numericas + columnas_categoricas]
y = df[objetivo]

print("\nDimensiones de X e y:")
print("X:", X.shape)
print("y:", y.shape)


Dimensiones de X e y:
X: (50000, 24)
y: (50000,)


In [14]:

# ============================================================
# 5. Preprocesamiento: imputación + escalado + OneHot
# ============================================================

# Numéricas: imputar mediana + escalar
transformador_numerico = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categóricas: imputar la moda + OneHot
transformador_categorico = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocesamiento = ColumnTransformer(
    transformers=[
        ("num", transformador_numerico, columnas_numericas),
        ("cat", transformador_categorico, columnas_categoricas)
    ]
)

In [15]:
# ============================================================
# 6. División entrenamiento / prueba
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # mantiene proporciones de clases
)


In [16]:
# ============================================================
# 7. Definir modelos de clasificación
# ============================================================
modelos = {
    "Regresión Logística": LogisticRegression(max_iter=1000),
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5),
    "Árbol de Decisión": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

In [17]:
# ============================================================
# 8. Entrenar, predecir y evaluar modelos
# ============================================================
resultados = []
mejor_modelo = None
mejor_nombre = None
mejor_accuracy = -1
mejor_pipeline = None

for nombre, modelo in modelos.items():
    print(f"\nEntrenando modelo: {nombre}")

    pipeline = Pipeline(steps=[
        ("preprocesamiento", preprocesamiento),
        ("modelo", modelo)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
    rec = recall_score(y_test, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

    resultados.append([nombre, acc, prec, rec, f1])

    if acc > mejor_accuracy:
        mejor_accuracy = acc
        mejor_modelo = modelo
        mejor_nombre = nombre
        mejor_pipeline = pipeline


Entrenando modelo: Regresión Logística

Entrenando modelo: KNN (k=5)

Entrenando modelo: Árbol de Decisión

Entrenando modelo: Random Forest

Entrenando modelo: Gradient Boosting


In [18]:
# ============================================================
# 9. Mostrar tabla de resultados
# ============================================================
df_resultados = pd.DataFrame(
    resultados,
    columns=["Modelo", "Accuracy", "Precision (weighted)", "Recall (weighted)", "F1 (weighted)"]
)

print("\nResultados de los modelos (ordenados por Accuracy):")
print(df_resultados.sort_values(by="Accuracy", ascending=False))


Resultados de los modelos (ordenados por Accuracy):
                Modelo  Accuracy  Precision (weighted)  Recall (weighted)  \
3        Random Forest    0.9794              0.974294             0.9794   
4    Gradient Boosting    0.9788              0.967756             0.9788   
0  Regresión Logística    0.9783              0.962070             0.9783   
1            KNN (k=5)    0.9773              0.964283             0.9773   
2    Árbol de Decisión    0.9640              0.966694             0.9640   

   F1 (weighted)  
3       0.969643  
4       0.969296  
0       0.968929  
1       0.969313  
2       0.965312  
